# PPO Synthetic vs Real: Multi-Test Experiment

This notebook is the interactive control panel and result report for the multi-window experiment. Long-running full training should normally run from a terminal; this notebook calls the same Python API when an interactive smoke run is useful.

The three training groups remain unchanged:

- real only;
- synthetic only, with equal probability across paths; and
- real + synthetic, with 50% real probability and the remaining 50% divided equally across synthetic paths.

Every checkpoint is selected on real validation data and evaluated on the matching untouched real test period.

## Recommended server workflow

Run training in a terminal, then reopen this notebook to inspect the saved suite:

```bash
python -m finrl.experiments.run_synthetic_vs_real \
  --config configs/synthetic_vs_real.json \
  --mode full
```

Use repeated `--window <name>` and `--ticker-group <name>` arguments to select experiment cases. The terminal runner writes `latest_suite.json`, which this notebook discovers automatically.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd


def bootstrap_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'finrl').is_dir() and (candidate / 'examples').is_dir():
            return candidate
    raise RuntimeError('Could not locate the FinRL project root.')


PROJECT_ROOT = bootstrap_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finrl.experiments.notebook_utils import environment_flag
from finrl.experiments.synthetic_vs_real import load_experiment_config
from finrl.experiments.synthetic_vs_real import run_experiment_suite
from finrl.experiments.synthetic_vs_real_plots import load_suite_artifacts
from finrl.experiments.synthetic_vs_real_plots import plot_learning_curves
from finrl.experiments.synthetic_vs_real_plots import plot_paired_metric_deltas
from finrl.experiments.synthetic_vs_real_plots import plot_representative_equity_curves
from finrl.experiments.synthetic_vs_real_plots import plot_representative_portfolio_weights
from finrl.experiments.synthetic_vs_real_plots import plot_test_metric_by_window

print('project root:', PROJECT_ROOT)

## 1. Configuration

Edit time splits, named ticker universes, and `active_ticker_groups` in `configs/synthetic_vs_real.json`. By default this notebook loads existing results. Set `FINRL_RUN_EXPERIMENT=1` only for an intentional interactive smoke or full run.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / os.getenv(
    'FINRL_EXPERIMENT_CONFIG', 'configs/synthetic_vs_real.json'
)
RUN_MODE = os.getenv('FINRL_RUN_MODE', 'smoke').lower()
RUN_EXPERIMENT = environment_flag('FINRL_RUN_EXPERIMENT', False)
USE_TRAIN_CACHE = environment_flag('FINRL_USE_TRAIN_CACHE', True)
FORCE_RETRAIN = environment_flag('FINRL_FORCE_RETRAIN', False)
SELECTED_WINDOWS = None  # Example: ['2024_train_val__2025_test']
SELECTED_TICKER_GROUPS = None  # Example: ['tech_5', 'chosen_10']

config = load_experiment_config(CONFIG_PATH)
settings = config.resolve_run_settings(RUN_MODE)
split_summary = pd.DataFrame(
    [
        {
            'window_id': split.name,
            'ticker_group': ticker_group,
            'tickers': ', '.join(tickers),
            'synthetic_period': split.synthetic_period_id,
            'train': f'[{split.train_start}, {split.train_end})',
            'validation': f'[{split.val_start}, {split.val_end})',
            'test': f'[{split.test_start}, {split.test_end})',
        }
        for split in config.select_splits(SELECTED_WINDOWS)
        for ticker_group, tickers in config.select_ticker_groups(
            SELECTED_TICKER_GROUPS
        )
    ]
)
print('mode:', RUN_MODE)
print('seeds:', settings.seeds)
print('timesteps:', f'{settings.total_timesteps:,}')
print('commissions:', settings.commission_map)
print('interactive execution:', RUN_EXPERIMENT)
display(split_summary)

## 2. Run or load the suite

When training is disabled, the notebook reads the latest terminal-produced suite for the selected mode.

In [ ]:
artifact_root = PROJECT_ROOT / config.artifact_dir
latest_suite_path = artifact_root / RUN_MODE / 'latest_suite.json'

if RUN_EXPERIMENT:
    suite_result = run_experiment_suite(
        project_root=PROJECT_ROOT,
        config=config,
        mode=RUN_MODE,
        split_names=SELECTED_WINDOWS,
        ticker_group_names=SELECTED_TICKER_GROUPS,
        use_cache=USE_TRAIN_CACHE,
        force_retrain=FORCE_RETRAIN,
        show_progress=True,
    )
    manifest_path = suite_result.manifest_path
elif latest_suite_path.is_file():
    manifest_path = latest_suite_path
else:
    manifest_path = None

if manifest_path is None:
    artifacts = None
    print('No saved suite found. Run the terminal command above first.')
else:
    artifacts = load_suite_artifacts(manifest_path)
    print('manifest:', artifacts.manifest_path)
    print('experiment cases:', artifacts.experiment_cases)

## 3. Data and test summaries

In [ ]:
if artifacts is not None:
    data_summaries = pd.concat(
        [
            artifacts.read_window_csv(
                window_id, 'data_summary.csv', ticker_group=ticker_group
            )
            for window_id, ticker_group in artifacts.experiment_cases
        ],
        ignore_index=True,
    )
    display(data_summaries)
    display(artifacts.aggregate_summary)

## 4. Cross-window comparison

The first figure summarizes seed dispersion; the second shows paired changes against the matching real-only run.

In [ ]:
PLOT_COMMISSION = 'with_fee'
PLOT_METRIC = 'sharpe'
PLOT_TICKER_GROUP = artifacts.ticker_groups[0] if artifacts is not None else None

if artifacts is not None:
    plot_test_metric_by_window(
        artifacts,
        metric=PLOT_METRIC,
        commission_name=PLOT_COMMISSION,
        ticker_group=PLOT_TICKER_GROUP,
    )
    plot_paired_metric_deltas(
        artifacts,
        metric=PLOT_METRIC,
        commission_name=PLOT_COMMISSION,
        ticker_group=PLOT_TICKER_GROUP,
    )

## 5. Learning, equity, and portfolio weights

Choose one time window and one training group for detailed inspection. Learning curves include each seed's smoothed episode reward and train/validation evaluation rewards; the equity and weight plots use the run nearest that group's median test Sharpe.

In [ ]:
if artifacts is not None:
    DETAIL_WINDOW = artifacts.window_ids[0]
    DETAIL_GROUP = 'real_trained'
    plot_learning_curves(
        artifacts,
        window_id=DETAIL_WINDOW,
        ticker_group=PLOT_TICKER_GROUP,
        group=DETAIL_GROUP,
        commission_name=PLOT_COMMISSION,
    )
    plot_representative_equity_curves(
        artifacts,
        window_id=DETAIL_WINDOW,
        ticker_group=PLOT_TICKER_GROUP,
        commission_name=PLOT_COMMISSION,
        period='test',
    )
    plot_representative_portfolio_weights(
        artifacts,
        window_id=DETAIL_WINDOW,
        ticker_group=PLOT_TICKER_GROUP,
        group=DETAIL_GROUP,
        commission_name=PLOT_COMMISSION,
    )

## Interpretation checklist

- Compare each synthetic or combined run with the same-window, same-fee, same-seed real-only policy.
- Treat synthetic validation as a diagnostic; checkpoint selection is based on real validation.
- Check whether improvements repeat across test windows instead of relying on one representative seed.
- Review turnover and portfolio weights together with return and Sharpe.
- Add new chronological windows and ticker groups to the JSON config before rerunning.
- Synthetic paths may use `data/synthetic/{train_start}_to_{val_end}/{ticker_group}/{dataset}/path_*.csv`; the existing period-level dataset layout remains supported as a fallback.